# 03 — Generics, TypeVars, Overloads, and Protocols

Use modern Python's static type system to describe relationships between values and to keep interfaces flexible without relying on runtime annotations for validation.

**Track:** Advanced Python · **Estimated lesson time:** 90–150 minutes · **Python:** 3.11+ (tested with 3.12) · **Prerequisites:** both prior levels.

## Objectives
- Preserve input/output relationships with TypeVar
- Build a generic container and discuss variance
- Declare behavior via Protocol
- Use @overload for meaningful alternative signatures

**How to learn:** Predict each result before running code, reproduce it, change one input, and explain the effect. Consider both normal behavior and failure paths. Restart the notebook kernel and run all cells before committing.

## Guided explanations and executable demonstrations

Each demonstration is deterministic, uses the Python standard library, and is safe to run locally without a live network service.

### 1. Generic functions

TypeVar describes a common type across parameters and results. On Python 3.11, this classic syntax is widely supported; newer type-parameter syntax is available in more recent versions.

In [ ]:
from typing import TypeVar
T = TypeVar("T")

def first_or_none(values: list[T]) -> T | None:
    return values[0] if values else None

print(first_or_none([1, 2]))
print(first_or_none(["a"]))
print(first_or_none([]))

**Design question:** Why does typing with T communicate more information than Any?

### 2. Generic mutable state

Generic classes can describe type-consistent operations. Mutable containers are usually invariant: permitting incompatible subtypes would undermine type safety during insertion.

In [ ]:
from typing import Generic, TypeVar
V = TypeVar("V")

class Stack(Generic[V]):
    def __init__(self):
        self._items: list[V] = []
    def push(self, value: V) -> None:
        self._items.append(value)
    def pop(self) -> V:
        return self._items.pop()

stack: Stack[int] = Stack()
stack.push(4)
stack.push(9)
print(stack.pop(), stack.pop())

**Design question:** What exception is appropriate when a stack has no values?

### 3. Structural protocols

Protocol defines an interface satisfied by behavior rather than inheritance. This makes it suitable for test doubles and adapters, though runtime validation remains the application's responsibility.

In [ ]:
from typing import Protocol
import json

class Serializer(Protocol):
    def dumps(self, value: dict) -> str: ...

class JsonSerializer:
    def dumps(self, value):
        return json.dumps(value, sort_keys=True)

def serialize(value: dict, serializer: Serializer) -> str:
    return serializer.dumps(value)

print(serialize({"b": 1, "a": 2}, JsonSerializer()))

**Design question:** How would a fake serializer simplify a service unit test?

### 4. Overloads

@overload signatures are static declarations; one implementation executes at runtime. Keep runtime branches consistent with documented types and reject unsupported values.

In [ ]:
from typing import overload

@overload
def double(value: int) -> int: ...
@overload
def double(value: str) -> str: ...

def double(value):
    if isinstance(value, int):
        return value * 2
    if isinstance(value, str):
        return value + value
    raise TypeError("expected int or str")

print(double(5), double("go"))

**Design question:** Why don't overload signatures themselves enforce runtime behavior?

## Independent practice

Do these before consulting the solutions. For each question, describe a contract, write a small implementation, test one ordinary case and one boundary/failure case, and note a trade-off.

**Exercise 1:** Write a generic last_or_none function with annotations.

In [ ]:
# Exercise 1
# TODO: Implement, demonstrate, and add assertions.

**Exercise 2:** Create a generic repository interface and a fake implementation.

In [ ]:
# Exercise 2
# TODO: Implement, demonstrate, and add assertions.

**Exercise 3:** Build a Protocol for a clock that returns a date.

In [ ]:
# Exercise 3
# TODO: Implement, demonstrate, and add assertions.

**Exercise 4:** Explain why a mutable list-like generic is usually invariant.

In [ ]:
# Exercise 4
# TODO: Implement, demonstrate, and add assertions.

## Common pitfalls and readiness check

- Static type hints do not validate untrusted JSON.
- Protocol-based design is useful when its interface is stable and small.
- Don't use Any when preserving a meaningful type relationship is possible.

**Reflect:** Which invariants does your implementation guarantee? How could you test them? What would change in production?

**Next:** Continue to the next numbered notebook and revisit the corresponding Advanced practice workbook.